# Functions and Graphs
**Topic:** Calculus for Machine Learning

In [1]:
import numpy as np
import pandas as pd

import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

import ipywidgets as widgets
from ipywidgets import interact, interactive, fixed, interact_manual
from ipywidgets import IntSlider, FloatSlider, Dropdown, Button, Output, HBox, VBox, Label

from IPython.display import display, HTML, clear_output
from scipy import stats
from tkh_utils import PALETTE, FONT, base_layout


---
## What you'll explore

By the end of this demo you will be able to:

- **Describe** a function as an input/output machine and read its behavior from its graph
- **Identify** the key features of common function shapes: linear, quadratic, exponential, and sigmoid
- **Interpret** how changing a function's parameters shifts, stretches, or flips its graph

> **Tip:** Start by selecting **sigmoid** from the function dropdown — that curve is the output of every logistic regression classifier you will train. Move the sliders and watch what changes before reading anything below.

---
## How we got here

This is the first notebook in the calculus series. In the statistics notebooks you worked with probability distributions, which are functions. In the Python notebooks you wrote functions in code. Here we zoom out and ask: what does it mean for a mathematical relationship to be a function, and what can the shape of its graph tell us?

---
## Why this matters for data science

Every machine learning model is a function: it takes inputs (your features) and produces an output (a prediction). The sigmoid function sits at the output layer of every logistic regression model and squashes any number into a value between 0 and 1, which is how probabilities are produced. Activation functions in neural networks, ReLU, tanh, sigmoid, are all functions chosen precisely for their shape. Recognizing these shapes is how you read a model's behavior at a glance.

---
## Try it yourself

In [ ]:
x = np.linspace(-5, 5, 400)

out = Output()

fn_dd = Dropdown(
    options=["Linear: ax + b", "Quadratic: a(x−b)²", "Cubic: ax³ + b",
             "Exponential: e^(ax)", "Logarithmic: a·ln(x+4)", "Sigmoid: 1/(1+e^(−ax+b))"],
    value="Sigmoid: 1/(1+e^(−ax+b))",
    description="Function:",
    style={"description_width": "85px"},
    layout=widgets.Layout(width="460px"))

a_slider = FloatSlider(value=1.0, min=-3.0, max=3.0, step=0.1,
    description="a:",
    style={"description_width": "30px"},
    layout=widgets.Layout(width="400px"))

b_slider = FloatSlider(value=0.0, min=-5.0, max=5.0, step=0.25,
    description="b:",
    style={"description_width": "30px"},
    layout=widgets.Layout(width="400px"))

def fn_compute(name, a, b, x):
    if name.startswith("Linear"):
        y   = a * x + b
        lab = f"f(x) = {a:.2f}x + {b:.2f}"
    elif name.startswith("Quadratic"):
        y   = a * (x - b) ** 2
        lab = f"f(x) = {a:.2f}(x − {b:.2f})²"
    elif name.startswith("Cubic"):
        y   = a * x ** 3 + b
        lab = f"f(x) = {a:.2f}x³ + {b:.2f}"
    elif name.startswith("Exp"):
        y   = np.exp(np.clip(a * x, -20, 20)) + b
        lab = f"f(x) = e^({a:.2f}x) + {b:.2f}"
    elif name.startswith("Log"):
        arg = x + 4.0
        y   = np.where(arg > 0, a * np.log(arg) + b, np.nan)
        lab = f"f(x) = {a:.2f}·ln(x+4) + {b:.2f}"
    else:  # Sigmoid
        y   = 1 / (1 + np.exp(np.clip(-a * x + b, -500, 500)))
        lab = f"f(x) = 1/(1+e^(−{a:.2f}x+{b:.2f}))"
    return y, lab

def render(fn_name, a, b):
    y, lab = fn_compute(fn_name, a, b, x)
    traces = [
        go.Scatter(x=[-5, 5], y=[0, 0], mode="lines",
            line=dict(color=PALETTE["muted"], width=1, dash="dot"), showlegend=False),
        go.Scatter(x=[0, 0], y=[-8, 8], mode="lines",
            line=dict(color=PALETTE["muted"], width=1, dash="dot"), showlegend=False),
        go.Scatter(x=x, y=np.clip(y, -8, 8), mode="lines",
            line=dict(color=PALETTE["primary"], width=2.5), name=lab),
    ]
    layout = base_layout(title=lab, xaxis_title="x", yaxis_title="f(x)")
    layout.update(xaxis=dict(range=[-5, 5]), yaxis=dict(range=[-8, 8]))
    with out:
        clear_output(wait=True)
        display(go.FigureWidget(go.Figure(data=traces, layout=layout)))

def on_change(change): render(fn_dd.value, a_slider.value, b_slider.value)
fn_dd.observe(on_change, names="value")
a_slider.observe(on_change, names="value")
b_slider.observe(on_change, names="value")
display(VBox([fn_dd, a_slider, b_slider, out]))
render(fn_dd.value, a_slider.value, b_slider.value)

---
## What's happening?

A function is a rule that takes an input, does something to it, and produces exactly one output. We write this as f(x), which reads "f of x" and means "apply the rule f to the value x." The graph of a function is just every input/output pair plotted as a point, connected into a curve.

| Function type | Shape | Key feature | ML where it appears |
|--------------|-------|-------------|---------------------|
| Linear: f(x) = mx + b | Straight line | Slope m controls steepness | Linear regression predictions |
| Quadratic: f(x) = ax² | Parabola | Has one turning point (minimum or maximum) | Mean squared error loss surface |
| Exponential: f(x) = eˣ | Rapid growth | Doubles repeatedly; never crosses zero | Softmax in multiclass classifiers |
| Logarithm: f(x) = ln(x) | Slow growth | Compresses large values | Log loss in logistic regression |
| Sigmoid: f(x) = 1/(1+e⁻ˣ) | S-shaped | Output always between 0 and 1 | Logistic regression output layer |
| ReLU: f(x) = max(0, x) | Flat then rising | Zero for negatives, linear for positives | Hidden layer activations in neural networks |

### Why the sigmoid is special

The sigmoid takes any number, no matter how large or small, and squashes it into the range (0, 1). That makes it perfect for representing a probability. The parameter inside the exponent controls how steep the S-shape is. Return to the widget and watch the sigmoid get steeper or shallower as you move the slope slider.

---
## Real-world example: Activation functions in a neural network

A neural network applies a function to the output of each layer. That function, called an activation function, is what allows the network to learn non-linear patterns. Without it, stacking layers would collapse into a single linear function, and the network could not model anything more complex than a straight line.

The chart below shows four common activation functions plotted side by side on the same input range. Notice:

- **Notice:** ReLU is zero for all negative inputs and linear for positive inputs; its sharp corner at zero makes it simple and fast to compute, which is why it is the default in most deep learning today
- **Notice:** The sigmoid saturates at 0 and 1 for very large and very small inputs; in those flat regions the derivative is nearly zero, which causes the "vanishing gradient" problem in deep networks
- **Notice:** Tanh is a rescaled sigmoid centered at zero; centering at zero makes optimization slightly better behaved than sigmoid

> **Discussion question:** A teammate proposes using a linear activation function (f(x) = x) in every layer of a neural network. Why would this make the network unable to learn any non-linear relationships, no matter how many layers you add?

In [3]:
x = np.linspace(-4, 4, 300)

activations = {
    "Sigmoid  f(x)=1/(1+e⁻ˣ)": (1 / (1 + np.exp(-x)),       PALETTE["primary"]),
    "Tanh  f(x)=tanh(x)":        (np.tanh(x),                  PALETTE["secondary"]),
    "ReLU  f(x)=max(0,x)":       (np.maximum(0, x),            PALETTE["accent"]),
    "Leaky ReLU  f(x)":          (np.where(x > 0, x, 0.1*x),  PALETTE["muted"]),
}

fig = go.Figure()
for name, (y, color) in activations.items():
    fig.add_trace(go.Scatter(x=x, y=y, mode="lines",
                             line=dict(color=color, width=2.5), name=name))
fig.add_hline(y=0, line_color=PALETTE["muted"], line_dash="dot", line_width=1)
fig.add_vline(x=0, line_color=PALETTE["muted"], line_dash="dot", line_width=1)
layout = base_layout(
    title="Neural Network Activation Functions — Shape Determines What a Layer Can Learn",
    xaxis_title="Input x", yaxis_title="f(x)",
)
layout.update(xaxis=dict(range=[-4, 4]), yaxis=dict(range=[-1.5, 1.5]))
fig.update_layout(layout)
fig.show()

### Common functions in machine learning

| Function type | Shape | ML where it appears | What it models |
|--------------|-------|---------------------|----------------|
| Linear f(x) = wx + b | Straight line | Linear and logistic regression | Direct proportional relationships |
| Sigmoid f(x) = 1/(1+e⁻ˣ) | S-curve, 0 to 1 | Logistic regression output, neural nets | Probability of binary outcome |
| ReLU f(x) = max(0,x) | Flat then linear | Neural network hidden layers | Non-linear feature transformations |
| Tanh f(x) = tanh(x) | S-curve, −1 to 1 | Neural network hidden layers | Centered activation |
| Softmax | Multiple sigmoids summing to 1 | Multiclass classifier output | Probability over multiple classes |
| Log f(x) = ln(x) | Slow growth | Log loss, log-likelihood | Penalizing confident wrong predictions |

---
## Key takeaway

> **A function is an input/output rule; its graph reveals its behavior at every input, and the shape you choose for activation functions or loss functions directly determines what a machine learning model can and cannot learn.**

---
*Next up: Limits & Continuity — the precise language for describing how a function behaves as the input approaches a point, and why that matters for gradient descent*